# ai_videogames· Transfer Learning para criaturas RPG
## Notebook guiado PRO · Oxford-IIIT Pet + MobileNetV2

### Objetivos
- Trabajar con un dataset visual más realista que MNIST/CIFAR-10
- Entender **transfer learning** y **fine-tuning**
- Usar un flujo moderno de visión por computador en Colab con GPU
- Conectar el problema con videojuegos: criaturas, companions, enemigos y mascotas RPG

### Qué haremos
1. Descargar y explorar **Oxford-IIIT Pet**
2. Preparar un pipeline `tf.data`
3. Entrenar un modelo preentrenado (fase 1: feature extraction)
4. Hacer **fine-tuning** (fase 2)
5. Evaluar cuantitativa y cualitativamente
6. Guardar el modelo final


## 0) Preparación
En Colab:
1. **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**
2. Ejecuta las celdas en orden
3. Guarda una copia en tu Drive si quieres conservar resultados


In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import os

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

IMG_SIZE = 160
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

## 1) Cargar el dataset
Usamos `tensorflow_datasets`, que descarga automáticamente **Oxford-IIIT Pet**.

Para clase, este dataset viene genial:
- es pequeño/medio
- tiene muchas clases
- es suficientemente realista
- permite enseñar técnicas de industria sin esperar horas


In [ ]:
(ds_train_raw, ds_test_raw), ds_info = tfds.load(
    "oxford_iiit_pet",
    split=["train[:80%]", "train[80%:]"],
    with_info=True,
    as_supervised=True
)

label_names = ds_info.features["label"].names
num_classes = ds_info.features["label"].num_classes

print(ds_info)
print("Número de clases:", num_classes)
print("Primeras clases:", label_names[:10])

## 2) Visualizar ejemplos
Regla básica en IA aplicada: **antes de entrenar, mira los datos**.


In [ ]:
plt.figure(figsize=(10,10))
for i, (image, label) in enumerate(ds_train_raw.take(9)):
    plt.subplot(3,3,i+1)
    plt.imshow(image)
    plt.title(label_names[int(label)], fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 3) Relectura 'videojuego'
Vamos a reinterpretar algunas clases como si fueran criaturas de un RPG.
Esto no cambia el dataset, solo su narrativa.


In [ ]:
RPG_ALIASES = {
    "abyssinian": "Felino del desierto",
    "american_bulldog": "Guardián pesado",
    "bengal": "Cazador sigiloso",
    "boxer": "Luchador de arena",
    "chihuahua": "Explorador pequeño",
    "egyptian_mau": "Gato místico",
    "great_pyrenees": "Tanque sagrado",
    "maine_coon": "Bestia nevada",
    "pomeranian": "Familiar mágico",
    "saint_bernard": "Guardián alpino",
}

def rpg_name(label_name):
    return RPG_ALIASES.get(label_name, f"Criatura {label_name}")

for name in label_names[:8]:
    print(name, "→", rpg_name(name))

## 4) Preprocesado
Usamos el preprocesado específico de **MobileNetV2**.
También montamos un pipeline `tf.data` eficiente.


In [ ]:
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = preprocess_input(tf.cast(image, tf.float32))
    return image, label

train_ds = (
    ds_train_raw
    .shuffle(2000, seed=SEED)
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = (
    ds_test_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

## 5) Data augmentation
Muy recomendable en visión por computador. Ayuda a generalizar mejor.


In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
], name="data_augmentation")

# Visualización rápida de augmentations
sample_image, sample_label = next(iter(ds_train_raw.take(1)))
plt.figure(figsize=(10,4))
for i in range(4):
    aug = data_augmentation(tf.expand_dims(tf.image.resize(sample_image, (IMG_SIZE, IMG_SIZE)), 0), training=True)
    plt.subplot(1,4,i+1)
    plt.imshow(tf.cast(aug[0], tf.uint8))
    plt.axis("off")
plt.suptitle("Ejemplos de data augmentation")
plt.show()

## 6) Base preentrenada
La idea clave del **transfer learning** es reutilizar una red ya entrenada en ImageNet.


In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False
print("Capas en la base:", len(base_model.layers))

## 7) Construir el modelo
Fase 1 = **feature extraction**:
- la base preentrenada está congelada
- solo entrenamos la 'cabeza' clasificadora


In [ ]:
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 8) Entrenar fase 1


In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=2,
        restore_best_weights=True
    )
]

initial_epochs = 5

history_1 = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=initial_epochs,
    callbacks=callbacks
)

## 9) Curvas de entrenamiento


In [ ]:
plt.figure()
plt.plot(history_1.history["accuracy"], label="train acc")
plt.plot(history_1.history["val_accuracy"], label="val acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Fase 1 · Feature extraction")
plt.legend()
plt.show()

## 10) Fine-tuning
Ahora desbloqueamos parte de la base y ajustamos con learning rate más pequeño.
Esto suele mejorar el resultado final.


In [ ]:
base_model.trainable = True

fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

fine_tune_epochs = 3
total_epochs = initial_epochs + fine_tune_epochs

history_2 = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=total_epochs,
    initial_epoch=history_1.epoch[-1] + 1,
    callbacks=callbacks
)

## 11) Curvas combinadas


In [ ]:
acc = history_1.history["accuracy"] + history_2.history["accuracy"]
val_acc = history_1.history["val_accuracy"] + history_2.history["val_accuracy"]

plt.figure()
plt.plot(acc, label="train acc")
plt.plot(val_acc, label="val acc")
plt.axvline(x=len(history_1.history["accuracy"])-1, linestyle="--", label="inicio fine-tuning")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Fase 1 + Fase 2")
plt.legend()
plt.show()

## 12) Evaluación final


In [ ]:
loss, acc = model.evaluate(test_ds, verbose=0)
print(f"Accuracy final: {acc:.4f}")

## 13) Matriz de confusión


In [ ]:
all_preds = []
all_true = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    all_preds.append(np.argmax(preds, axis=1))
    all_true.append(labels.numpy())

all_preds = np.concatenate(all_preds)
all_true = np.concatenate(all_true)

cm = tf.math.confusion_matrix(all_true, all_preds, num_classes=num_classes).numpy()

plt.figure(figsize=(8,7))
plt.imshow(cm)
plt.title("Matriz de confusión")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.colorbar()
plt.show()

## 14) Aciertos y errores
La evaluación visual suele enseñar mucho más que una métrica sola.


In [ ]:
# Recolectamos algunas muestras del test
sample_images = []
sample_labels = []
for images, labels in test_ds.take(3):
    sample_images.append(images)
    sample_labels.append(labels)

sample_images = tf.concat(sample_images, axis=0)
sample_labels = tf.concat(sample_labels, axis=0)

preds = model.predict(sample_images, verbose=0)
pred_classes = np.argmax(preds, axis=1)

correct = np.where(pred_classes == sample_labels.numpy())[0]
wrong = np.where(pred_classes != sample_labels.numpy())[0]

plt.figure(figsize=(12,8))
for i, idx in enumerate(correct[:12]):
    plt.subplot(3,4,i+1)
    img = (sample_images[idx].numpy() + 1.0) / 2.0
    plt.imshow(np.clip(img, 0, 1))
    plt.title(f"OK\n{label_names[int(pred_classes[idx])]}", fontsize=8)
    plt.axis("off")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12,8))
for i, idx in enumerate(wrong[:12]):
    plt.subplot(3,4,i+1)
    img = (sample_images[idx].numpy() + 1.0) / 2.0
    plt.imshow(np.clip(img, 0, 1))
    t = label_names[int(sample_labels[idx])]
    p = label_names[int(pred_classes[idx])]
    plt.title(f"T:{t}\nP:{p}", fontsize=8)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 15) Guardar modelo


In [ ]:
model.save("semana3_pro_transfer_learning_oxfordpets.keras")
print("Modelo guardado.")

## ✅ Ideas clave de la semana
- **Transfer learning** ahorra tiempo y datos
- **Fine-tuning** puede mejorar el ajuste al dominio objetivo
- Esta técnica es mucho más realista que entrenar CNN grandes desde cero
- En videojuegos, este patrón sirve para criaturas, enemigos, mascotas y otros assets visuales
